# Feature Selection for Classification

**Feature selection** means choosing which features to use in our model.

Why do it?

- Reduce overfitting: when a model fits the training data very well but does not work as well on new data.
- Make models simpler and faster.
- Remove features that are not useful.

We will study three approaches:

1. **Filter methods:** score each feature using the data.
2. **Model-based methods:** use a model to select features.
3. **Wrapper methods:** fit models using different sets of features.

## 1. Imports

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import (
    SelectKBest,
    mutual_info_classif,
    SelectFromModel,
    RFE,
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    average_precision_score,
    roc_auc_score,
    classification_report,
)


## 2. Load the Bank Marketing data
We will use the **Bank Marketing** dataset. It contains numerical and categorical variables. We will also create extra features using polynomial features and one-hot encoding.


The target tells us whether a client subscribed to a term deposit.The Bank Marketing dataset contains information about clients contacted by a bank. The goal is to predict whether a client will subscribe to a term deposit.ing.

In [4]:
path = 'https://raw.githubusercontent.com/um-perez-alvaro/Data-Science-Practice/refs/heads/master/Data/bank_marketing.cvs'
df = pd.read_csv(path)
df

,index,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,21412,34,management,single,tertiary,no,757,no,no,cellular,18,aug,37,4,-1,0,unknown,0
1,41175,39,technician,single,tertiary,no,139,no,no,telephone,19,aug,328,1,79,1,failure,0
2,18456,44,management,married,tertiary,no,4460,no,no,telephone,31,jul,168,2,-1,0,unknown,0
3,15791,25,admin.,single,secondary,no,105,no,yes,cellular,21,jul,392,1,-1,0,unknown,0
4,4250,44,management,divorced,tertiary,no,409,yes,no,unknown,19,may,333,1,-1,0,unknown,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5995,24853,33,management,married,tertiary,no,0,yes,no,cellular,18,nov,88,1,-1,0,unknown,0
5996,35629,43,management,married,tertiary,no,9,yes,no,telephone,8,may,142,3,358,3,failure,0
5997,493,55,unemployed,married,tertiary,no,5345,no,no,unknown,6,may,278,1,-1,0,unknown,0
5998,10367,40,blue-collar,married,secondary,no,952,no,yes,unknown,12,jun,140,1,-1,0,unknown,0


`duration` is the length of the call. We only know it **after the call**.

If we want to predict whom to call, we cannot use `duration`. This is an example of **data leakage**.

In [5]:
X = df.drop(['label','duration'],axis=1)
y = df.label

In [6]:
# imbalanced dataset
y.value_counts(dropna=False)

label
0    5298
1     702
Name: count, dtype: int64

In [ ]:
 X = X.drop(columns="duration")

In [ ]:
len(X)

### Add some random features

We will add some random features that we **know are not useful** for predicting the target.

A good feature-selection method should remove these features.

In [7]:
# Add random features
X["random_1"] = np.random.normal(size=len(X))
X["random_2"] = np.random.normal(size=len(X))
X["random_category"] = np.random.choice(
    ["A", "B", "C", "D"], size=len(X)
)

print("Shape:", X.shape)
print("Positive class:", y.mean())

Shape: (6000, 19)
Positive class: 0.117


## 3. Train/test split

We split **before** fitting preprocessing or feature-selection steps. The test set will remain untouched until we evaluate a fitted model.

In [8]:
# Keep approximately the same proportion of each class in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y
)

print("Training observations:", len(X_train))
print("Test observations:    ", len(X_test))

Training observations: 4500
Test observations:     1500


## 4. Preprocessing pipeline

We will:

- impute and standardize numerical variables;
- impute and one-hot encode categorical variables.


In [10]:
num_features = ['index',
 'age',
 'balance',
 'day',
 'campaign',
 'pdays',
 'previous',
 'random_1',
 'random_2']
cat_features = ['job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'poutcome',
 'random_category']


print("Numerical columns:", len(num_features))
print("Categorical columns:", len(cat_features))
print()
print("Categorical columns:")
print(cat_features)

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

processor = ColumnTransformer([
    ("num", num_pipe, num_features),
    ("cat", cat_pipe, cat_features),
])
processor

Numerical columns: 9
Categorical columns: 10

Categorical columns:
['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'random_category']


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['index', 'age', 'balance', 'day', 'campaign',
                                  'pdays', 'previous', 'random_1',
                                  'random_2']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'month',
                                  'poutcome', 'random_category'])])

### How many features did one-hot encoding create?

In [14]:
processor.fit(X_train)
encoded_names = processor.get_feature_names_out()

print("Raw input columns:       ", X_train.shape[1])## 5. Create more features

print("After preprocessing:     ", len(encoded_names))

pd.DataFrame({"encoded_feature": encoded_names})

Raw input columns:        19
After preprocessing:      57


,encoded_feature
0,num__index
1,num__age
2,num__balance
3,num__day
4,num__campaign
5,num__pdays
6,num__previous
7,num__random_1
8,num__random_2
9,cat__job_admin.


## 5. Create more features

We use degree-2 polynomial features to create new features and interactions between features.

This can quickly create **many features**.

In [16]:
poly = PolynomialFeatures(degree=2, include_bias=False)

X_train_encoded = processor.transform(X_train)
poly.fit(X_train_encoded)

poly_names = poly.get_feature_names_out(encoded_names)

print("Raw input columns:   ", X_train.shape[1])
print("Encoded features:    ", len(encoded_names))
print("Degree-2 features:   ", len(poly_names))

Raw input columns:    19
Encoded features:     57
Degree-2 features:    1710


### Look at the resulting features

Do not try to interpret every feature. The point is to see how quickly a modest raw dataset can become a high-dimensional modeling problem.

In [17]:
feature_table = pd.DataFrame({"feature": poly_names})
feature_table

,feature
0,num__index
1,num__age
2,num__balance
3,num__day
4,num__campaign
...,...
1705,cat__random_category_B cat__random_category_C
1706,cat__random_category_B cat__random_category_D
1707,cat__random_category_C^2
1708,cat__random_category_C cat__random_category_D


## 6. Logistic regression with many features

We fit logistic regression using all the degree-2 features.

We compare the **training** and **test** results. If the model does much better on the training data, it may be overfitting.

Since the classes are imbalanced, we look at several metrics.

In [18]:
y_train.value_counts()

label
0    3974
1     526
Name: count, dtype: int64

In [20]:
pipe = Pipeline([
    ("preprocess", processor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("logreg", LogisticRegression(max_iter=3000))
])

pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['index', 'age', 'balance',
                                                   'day', 'campaign', 'pdays',
                                                   'previous', 'random_1',
                                                   'random_2']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome',
                                                   'random_category'])])),
                ('poly', PolynomialFeatures(include_bias=False)),
                ('logreg', LogisticRegression(max_iter=3000))])

In [21]:
y_train_pred = pipe.predict(X_train)
y_train_prob = pipe.predict_proba(X_train)[:, 1]

y_test_pred = pipe.predict(X_test)
y_test_prob = pipe.predict_proba(X_test)[:, 1]

print("TRAIN")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("ROC AUC:", roc_auc_score(y_train, y_train_prob))
print("Average precision:", average_precision_score(y_train, y_train_prob))

print("\nTEST")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("ROC AUC:", roc_auc_score(y_test, y_test_prob))
print("Average precision:", average_precision_score(y_test, y_test_prob))

TRAIN
Accuracy: 0.9448888888888889
ROC AUC: 0.9425763661518503
Average precision: 0.8088330528153113

TEST
Accuracy: 0.8706666666666667
ROC AUC: 0.7260925913210656
Average precision: 0.36339047793813817


### Conclusions

The model performs better on the training data than on the test data. This is **overfitting**.

Having many features can make overfitting worse. Some features may also be useless.

This motivates **feature selection**: keeping the useful features and removing the others.

# Approach 1: Filter methods

A **filter method** gives each feature a score and keeps the best ones.

We use:

```python
SelectKBest(score_func=mutual_info_classif, k=20)
```

There are two main parameters:

- `score_func`: how we score each feature.
- `k`: how many features we keep.

For classification, some common choices for `score_func` are:

- `f_classif`: compares the **average value** of the feature across the classes. A high score means the class averages are different compared with the variation within each class. It is simple and fast.

- `chi2`: compares the **observed values** with what we would expect if the feature and the target were independent. A high score means there is evidence of an association between the feature and the target. It requires nonnegative features.

- `mutual_info_classif`: measures **how much knowing the feature tells us about the target**. It is more flexible because it can detect both linear and nonlinear relationships.

Mutual information is often a good general choice.

- **High mutual information:** knowing the feature gives us useful information about the target.
- **Low mutual information:** knowing the feature tells us little about the target.
- **Zero mutual information:** the feature and the target are independent.

Here, we use `mutual_info_classif` and keep the 20 features with the highest scores.

In [ ]:
# We keep the 20 features with the highest scores
# Feature selection should be part of the pipeline.
pipe = Pipeline([
    ("preprocess", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("select", SelectKBest(score_func=mutual_info_classif, k=20)),
    ("logreg", LogisticRegression(max_iter=3000, solver="liblinear")),
])

pipe

The number of features to keep, $k$, is a **hyperparameter**.

We could choose it using cross-validation. We will not do this today because it is a bit slow.n.

In [ ]:
# This may be slow because we compute mutual information for many features
# We only need to do this once.
pipe.fit(X_train,y_train)

### Which features were selected?

In [ ]:
# Names after preprocessing
feature_names = pipe["preprocess"].get_feature_names_out()

# Names after polynomial features
feature_names = pipe["poly"].get_feature_names_out(feature_names)

len(feature_names)

In [ ]:
# Which features were selected?
selected = pipe["select"].get_support()
feature_names[selected]

### Did we remove all the random features?

No. Some interactions with random features were selected.

Random features can appear useful just by chance, especially when we create many features.

Feature selection does not guarantee that every selected feature is truly useful.

In [ ]:
# Training
y_train_pred = pipe.predict(X_train)
y_train_prob = pipe.predict_proba(X_train)[:, 1]

# Test
y_test_pred = pipe.predict(X_test)
y_test_prob = pipe.predict_proba(X_test)[:, 1]

print("TRAIN")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_train, y_train_pred))
print("ROC AUC:", roc_auc_score(y_train, y_train_prob))
print("Average precision:", average_precision_score(y_train, y_train_prob))

print("\nTEST")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred))
print("ROC AUC:", roc_auc_score(y_test, y_test_prob))
print("Average precision:", average_precision_score(y_test, y_test_prob))

### Conclusions

- There is some **overfitting**, but not too much.
- The ROC AUC and average precision show that the model is **not very strong**.
- The data are imbalanced, so we could improve the classification by **changing the threshold**.
- We will not do threshold tuning today.

# Approach 2: Model-based selection

A **model-based method** fits a model and uses what the model learns to select features.

For logistic regression, we can use the **coefficients**.

Recall that

$$
p = \frac{1}{1+e^{-z}},
\qquad
z = \beta_0+\beta_1x_1+\cdots+\beta_mx_m.
$$

The numbers $\beta_1,\ldots,\beta_m$ are the **coefficients** learned by logistic regression. They determine how each feature affects the predicted probability $p$.

Recall that logistic regression minimizes the loss

$$
-\sum_{i=1}^n
\left[
y_i\log(p_i)+(1-y_i)\log(1-p_i)
\right],
$$

where $p_i$ is the predicted probability that $y_i=1$.

## Regularization

We can modify the loss by adding a **penalty** for large coefficients. This is called **regularization**.

Two common choices are **L1** and **L2 regularization**.

With **L1 regularization**, we minimize

$$
-\sum_{i=1}^n
\left[
y_i\log(p_i)+(1-y_i)\log(1-p_i)
\right]
+
\lambda\sum_{j=1}^m |\beta_j|.
$$

With **L2 regularization**, we minimize

$$
-\sum_{i=1}^n
\left[
y_i\log(p_i)+(1-y_i)\log(1-p_i)
\right]
+
\lambda\sum_{j=1}^m \beta_j^2.
$$

In both cases:

- The first term asks the model to **fit the data well**.
- The second term asks the model to **keep the coefficients small**.
- $\lambda$ controls how strong the penalty is.

The main difference is:

- **L1** can make some coefficients exactly zero.
- **L2** makes coefficients smaller, but usually not exactly zero.

For feature selection, **L1 is especially useful** because a zero coefficient means that the corresponding feature is not used by the model.


If $\beta_j=0$, then feature $j$ does not contribute to the prediction.
We can remove that feature.

This gives us a way to do **feature selection** with logistic regression.

`SelectFromModel` does this for us: it fits the model and keeps the features with important coefficients.

In [47]:
pipe = Pipeline([
    ("preprocess", processor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),

    # Fit an L1 logistic regression
    # and remove features with zero coefficients
    ("select", SelectFromModel(
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.05,              # Smaller C = stronger regularization
            max_iter=3000,
        ),
        threshold=1e-8,         # Keep coefficients that are not zero
    )),

    # Fit the final logistic regression using the selected features
    ("logreg", LogisticRegression(max_iter=3000, solver="liblinear")),
])

pipe

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['index', 'age', 'balance',
                                                   'day', 'campaign', 'pdays',
                                                   'previous', 'random_1',
                                                   'random_2']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle...),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome',
                                                   'random_category'])])),
                ('poly', PolynomialFeatures(include_bias=False)),
                ('select',
                 SelectFromModel(estimator=LogisticRegression(C=0.05,
                                                              max_iter=3000,
                                                              penalty='l1',
                                                              solver='liblinear'),
                                 threshold=1e-08)),
                ('logreg',
                 LogisticRegression(max_iter=3000, solver='liblinear'))])

In [48]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['index', 'age', 'balance',
                                                   'day', 'campaign', 'pdays',
                                                   'previous', 'random_1',
                                                   'random_2']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle...),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month', 'poutcome',
                                                   'random_category'])])),
                ('poly', PolynomialFeatures(include_bias=False)),
                ('select',
                 SelectFromModel(estimator=LogisticRegression(C=0.05,
                                                              max_iter=3000,
                                                              penalty='l1',
                                                              solver='liblinear'),
                                 threshold=1e-08)),
                ('logreg',
                 LogisticRegression(max_iter=3000, solver='liblinear'))])

In [49]:
# Feature names after preprocessing
feature_names = pipe["preprocess"].get_feature_names_out()

# Feature names after polynomial features
feature_names = pipe["poly"].get_feature_names_out(feature_names)

# Features kept by SelectFromModel
selected = pipe["select"].get_support()

selected_features = feature_names[selected]

print("Number of features:", len(feature_names))
print("Number selected:", len(selected_features))

selected_features

Number of features: 1710
Number selected: 44


array(['num__index', 'num__random_2', 'cat__marital_married',
       'cat__housing_yes', 'cat__month_may', 'num__index^2',
       'num__index num__age', 'num__index num__balance',
       'num__index num__campaign', 'num__index num__pdays',
       'num__index num__random_2', 'num__index cat__default_no',
       'num__index cat__housing_no', 'num__index cat__loan_no',
       'num__index cat__contact_cellular',
       'num__index cat__poutcome_success', 'num__age num__balance',
       'num__age num__previous', 'num__age num__random_1',
       'num__age cat__random_category_C',
       'num__age cat__random_category_D', 'num__balance^2',
       'num__balance num__previous', 'num__balance cat__poutcome_unknown',
       'num__day^2', 'num__day num__previous', 'num__day cat__month_feb',
       'num__campaign^2', 'num__campaign num__previous',
       'num__campaign cat__education_secondary', 'num__pdays^2',
       'num__pdays num__random_2', 'num__pdays cat__job_blue-collar',
       'num__pdays

In [50]:
# Training
y_train_pred = pipe.predict(X_train)
y_train_prob = pipe.predict_proba(X_train)[:, 1]

# Test
y_test_pred = pipe.predict(X_test)
y_test_prob = pipe.predict_proba(X_test)[:, 1]

print("TRAIN")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_train, y_train_pred))
print("ROC AUC:", roc_auc_score(y_train, y_train_prob))
print("Average precision:", average_precision_score(y_train, y_train_prob))

print("\nTEST")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred))
print("ROC AUC:", roc_auc_score(y_test, y_test_prob))
print("Average precision:", average_precision_score(y_test, y_test_prob))

TRAIN
Accuracy: 0.8964444444444445
Balanced accuracy: 0.6098183822220862
ROC AUC: 0.8075451461113204
Average precision: 0.4668321362304532

TEST
Accuracy: 0.894
Balanced accuracy: 0.6098770942048888
ROC AUC: 0.8022778769568799
Average precision: 0.4737802891854511


## Tune the selector

The strength of the L1 penalty is a **hyperparameter**.

`select__estimator__C` controls this penalty. Smaller `C` means stronger regularization, so we usually select **fewer features**.

We can choose `C` using cross-validation.

In [51]:
model_grid = {
    # C for the logistic regression used to select features
    "select__estimator__C": [0.01,0.05, 0.1, 1],

    # Which coefficients are large enough to keep
    "select__threshold": [1e-8, "median"],

    # C for the final logistic regression
    "logreg__C": [0.1, 1, 10],
}



There are two logistic regression models in our pipeline:

- `select__estimator__C`: controls the L1 regularization used to **select features**. It affects which coefficients become zero.
- `select__threshold`: decides which coefficients we keep. `1e-8` keeps essentially all nonzero coefficients. `"median"` keeps features with coefficients at least as large as the median.
- `logreg__C`: controls the regularization of the **final logistic regression**, after the features have been selected.

We can choose these values using cross-validation.

In [52]:
model_search = GridSearchCV(
    model_based_pipe,
    param_grid=model_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
)

model_search.fit(X_train, y_train)

print("Best parameters:", model_search.best_params_)
print("Best CV average precision:", round(model_search.best_score_, 3))

Best parameters: {'logreg__C': 10, 'select__estimator__C': 0.05, 'select__threshold': 1e-08}
Best CV average precision: 0.404


NameError: name 'evaluate_model' is not defined

### Inspect the selected features

In [53]:
best_model_based = model_search.best_estimator_

pre = best_model_based.named_steps["preprocess"]
poly_step = best_model_based.named_steps["poly"]
selector = best_model_based.named_steps["select"]

names_1 = pre.get_feature_names_out()
names_2 = poly_step.get_feature_names_out(names_1)
selected_model_names = names_2[selector.get_support()]

print("Features before selection:", len(names_2))
print("Features after selection: ", len(selected_model_names))

pd.DataFrame({"selected_feature": selected_model_names}).head(100)

Features before selection: 1710
Features after selection:  45


,selected_feature
0,num__index
1,num__random_2
2,cat__marital_married
3,cat__housing_yes
4,cat__month_may
5,num__index^2
6,num__index num__age
7,num__index num__balance
8,num__index num__campaign
9,num__index num__pdays


In [54]:
# Training
y_train_pred = best_model_based.predict(X_train)
y_train_prob = best_model_based.predict_proba(X_train)[:, 1]

# Test
y_test_pred = best_model_based.predict(X_test)
y_test_prob = best_model_based.predict_proba(X_test)[:, 1]

print("TRAIN")
print("Accuracy:", accuracy_score(y_train, y_train_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_train, y_train_pred))
print("ROC AUC:", roc_auc_score(y_train, y_train_prob))
print("Average precision:", average_precision_score(y_train, y_train_prob))

print("\nTEST")
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred))
print("ROC AUC:", roc_auc_score(y_test, y_test_prob))
print("Average precision:", average_precision_score(y_test, y_test_prob))

TRAIN
Accuracy: 0.8964444444444445
Balanced accuracy: 0.6089936296956835
ROC AUC: 0.8074585566639431
Average precision: 0.46716574586197956

TEST
Accuracy: 0.894
Balanced accuracy: 0.6098770942048888
ROC AUC: 0.8023765792364735
Average precision: 0.47505706210713683


# Approach 3: Wrapper methods

A **wrapper method** uses model fitting itself to search for a useful subset of features.

We will use **recursive feature elimination (RFE)**:

1. fit the model;
2. identify the least important features;
3. remove some of them;
4. fit the model again;
5. repeat.

This can be much more expensive than a filter method because the model is fitted many times.

For that reason, we first demonstrate RFE on the **encoded degree-1 feature space**, rather than on the thousands of degree-2 features.

In [ ]:
rfe_estimator = LogisticRegression(max_iter=3000, solver="liblinear")

wrapper_pipe = Pipeline([
    ("preprocess", preprocessor),
    ("select", RFE(
        estimator=rfe_estimator,
        n_features_to_select=20,
        step=0.2,
    )),
    ("logreg", LogisticRegression(max_iter=3000, solver="liblinear")),
])

wrapper_pipe

## Tune the number of features kept by RFE

In [ ]:
wrapper_grid = {
    "select__n_features_to_select": [10, 20, 40],
    "logreg__C": [0.1, 1, 10],
}

wrapper_search = GridSearchCV(
    wrapper_pipe,
    param_grid=wrapper_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
)

wrapper_search.fit(X_train, y_train)

print("Best parameters:", wrapper_search.best_params_)
print("Best CV average precision:", round(wrapper_search.best_score_, 3))

evaluate_model(wrapper_search.best_estimator_, X_train, y_train, X_test, y_test)

### Inspect the RFE-selected features

In [ ]:
best_wrapper = wrapper_search.best_estimator_

encoded_names_wrapper = best_wrapper.named_steps["preprocess"].get_feature_names_out()
wrapper_selector = best_wrapper.named_steps["select"]
selected_wrapper_names = encoded_names_wrapper[wrapper_selector.get_support()]

pd.DataFrame({"selected_feature": selected_wrapper_names})

## Can RFE be used with the degree-2 feature space?

Yes. The pipeline would be

```python
Pipeline([
    ("preprocess", preprocessor),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("select", RFE(...)),
    ("logreg", LogisticRegression(...)),
])
```

But now RFE starts with thousands of features and repeatedly refits logistic regression. This can be slow.

This highlights a practical difference among the three approaches:

- **Filter methods** are usually cheap.
- **Model-based methods** require fitting a model, but usually only once per parameter setting.
- **Wrapper methods** repeatedly fit models and can become expensive in high dimensions.

A common practical strategy is to use a cheap filter first to reduce a very large feature space, and then apply a more expensive method to the smaller set.

# 10. Compare the methods

Let's collect the test performance of the large model and the three feature-selection approaches.

In [ ]:
models = {
    "All degree-2 features": big_model,
    "Filter: SelectKBest": filter_search.best_estimator_,
    "Model-based: L1": model_search.best_estimator_,
    "Wrapper: RFE (degree 1)": wrapper_search.best_estimator_,
}

comparison = []
for name, model in models.items():
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    comparison.append({
        "model": name,
        "test_accuracy": accuracy_score(y_test, pred),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "test_ROC_AUC": roc_auc_score(y_test, prob),
        "test_average_precision": average_precision_score(y_test, prob),
    })

comparison = pd.DataFrame(comparison).set_index("model")
comparison.round(3)

# 11. Main ideas

### Why select features?

Feature selection can:

- reduce overfitting;
- make models faster;
- simplify interpretation;
- remove irrelevant or redundant variables.

But **more feature selection is not automatically better**. Removing useful information can hurt prediction.

### Three approaches

| Approach | Main idea | Example used here |
|---|---|---|
| Filter | Score features without repeatedly fitting the final model | `SelectKBest(mutual_info_classif)` |
| Model-based | Use information learned by a fitted model | `SelectFromModel` + L1 logistic regression |
| Wrapper | Fit models repeatedly while searching over subsets | `RFE` |

### The most important workflow rule

Do **not** select features using the full dataset and then cross-validate the classifier.

Instead, put feature selection inside the pipeline:

\[
\text{preprocessing}
\to
\text{feature construction}
\to
\text{feature selection}
\to
\text{classifier}.
\]

Then cross-validation repeats the **entire modeling process** separately inside each training fold.

# 12. Questions to think about

1. Did the degree-2 model overfit? Which metrics make this most visible?
2. How many degree-2 features did we create from the original columns?
3. Why must feature selection happen inside the pipeline?
4. Which method selected the fewest features without substantially hurting test performance?
5. Did any method select features involving our deliberately random variables?
6. Why is RFE much more expensive than `SelectKBest`?
7. Would you always choose the method with the highest test score? What else might matter?
8. What happens if you change the sample size? Does the large degree-2 model overfit more or less?
9. What happens if you remove the deliberately random features?
10. What happens if you use `interaction_only=True` in `PolynomialFeatures`?